# EfficientNet with Focal Loss

This notebook implements Focal Loss to handle class imbalance:
- Focal Loss focuses on hard examples
- Reduces the relative loss for well-classified examples
- Prevents easy negatives from overwhelming the training
- Particularly useful for imbalanced datasets

In [1]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
import os
import sys
sys.path.append('../../..')
from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint, evaluation, format_metrics
from utils.train import train_model
from utils.models import EfficientNetApi, EfficientNetApiGem

In [2]:
seed = 42
batch_size = 6
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'
data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


## Focal Loss Implementation

In [3]:
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: Weighting factor in range (0,1) to balance positive/negative examples
        gamma: Exponent of the modulating factor (1 - p_t)^gamma
               Higher gamma focuses more on hard examples
        reduction: 'mean', 'sum' or 'none'
    """
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        # Apply sigmoid to get probabilities
        p = torch.sigmoid(inputs)
        
        # Calculate binary cross entropy
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        
        # Calculate p_t
        p_t = p * targets + (1 - p) * (1 - targets)
        
        # Calculate focal loss
        focal_weight = (1 - p_t) ** self.gamma
        focal_loss = self.alpha * focal_weight * bce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class AdaptiveFocalLoss(nn.Module):
    """
    Adaptive Focal Loss with learnable alpha parameter
    """
    def __init__(self, gamma=2.0, reduction='mean'):
        super(AdaptiveFocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
        # Learnable alpha parameter
        self.alpha = nn.Parameter(torch.tensor([0.25]))
    
    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs)
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        p_t = p * targets + (1 - p) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        focal_loss = torch.sigmoid(self.alpha) * focal_weight * bce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

print("Focal Loss implementations created")

Focal Loss implementations created


In [4]:
load_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

## Load Dataset

In [5]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train_.columns = df_train_.columns.str.strip()
train_indexes = np.where((df_train_['fold'] != 3))[0]
valid_indexes = np.where((df_train_['fold'] == 3))[0]

df_train = df_train_.loc[train_indexes]
df_val = df_train_.loc[valid_indexes]
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

# Check class distribution
print("\nClass distribution in training set:")
print(df_train['isup_grade'].value_counts().sort_index())
print("\nClass distribution in validation set:")
print(df_val['isup_grade'].value_counts().sort_index())


Class distribution in training set:
isup_grade
0    1966
1    1813
2     913
3     845
4     850
5     832
Name: count, dtype: int64

Class distribution in validation set:
isup_grade
0    492
1    453
2    229
3    211
4    212
5    208
Name: count, dtype: int64


In [6]:
transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms)
valid_dataset = PandasDataset(images_dir, df_val, transforms=None)
test_dataset = PandasDataset(images_dir, df_test, transforms=None)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(test_dataset)
)

## Training with Focal Loss (gamma=2.0)

In [7]:
# Use Focal Loss with gamma=2.0 (standard)
loss_function = FocalLoss(alpha=0.25, gamma=2.0)

optimizer = optim.Adam(model.parameters(), lr=init_lr/warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier=warmup_factor, total_epoch=warmup_epochs, after_scheduler=scheduler_cosine)

print("\n=== Training with Focal Loss (gamma=2.0) ===")
train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/focal-loss-gamma2.txt",
    path_to_save_model="models/focal-loss-gamma2.pth",
    patience=5,
)


=== Training with Focal Loss (gamma=2.0) ===
Epoch 1/50



100%|██████████| 301/301 [01:37<00:00,  3.09it/s]


VAL_LOSS     0.019
VAL_ACC      Mean: 29.270 | Std: 1.111 | 95% CI: [27.424, 31.080]
VAL_KAPPA    Mean: 0.622 | Std: 0.011 | 95% CI: [0.604, 0.640]
VAL_F1       Mean: 0.237 | Std: 0.009 | 95% CI: [0.223, 0.251]
VAL_RECALL   Mean: 0.324 | Std: 0.010 | 95% CI: [0.306, 0.340]
VAL_PRECISION Mean: 0.308 | Std: 0.081 | 95% CI: [0.193, 0.382]
Salvando o melhor modelo... 0.0 -> 0.6224119867319073
Epoch 2/50



100%|██████████| 301/301 [01:40<00:00,  3.01it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.020
VAL_ACC      Mean: 28.131 | Std: 1.039 | 95% CI: [26.482, 29.809]
VAL_KAPPA    Mean: 0.594 | Std: 0.011 | 95% CI: [0.576, 0.611]
VAL_F1       Mean: 0.196 | Std: 0.007 | 95% CI: [0.184, 0.207]
VAL_RECALL   Mean: 0.292 | Std: 0.009 | 95% CI: [0.278, 0.306]
VAL_PRECISION Mean: 0.307 | Std: 0.056 | 95% CI: [0.159, 0.342]
Epoch 3/50



...

VAL_LOSS     0.048
VAL_ACC      Mean: 58.980 | Std: 1.164 | 95% CI: [57.119, 60.997]
VAL_KAPPA    Mean: 0.821 | Std: 0.011 | 95% CI: [0.802, 0.839]
VAL_F1       Mean: 0.537 | Std: 0.012 | 95% CI: [0.517, 0.557]
VAL_RECALL   Mean: 0.540 | Std: 0.012 | 95% CI: [0.520, 0.560]
VAL_PRECISION Mean: 0.570 | Std: 0.012 | 95% CI: [0.551, 0.590]
Epoch 13/50



100%|██████████| 301/301 [01:37<00:00,  3.09it/s]


VAL_LOSS     0.047
VAL_ACC      Mean: 57.575 | Std: 1.123 | 95% CI: [55.679, 59.446]
VAL_KAPPA    Mean: 0.813 | Std: 0.011 | 95% CI: [0.795, 0.831]
VAL_F1       Mean: 0.534 | Std: 0.012 | 95% CI: [0.515, 0.554]
VAL_RECALL   Mean: 0.537 | Std: 0.012 | 95% CI: [0.516, 0.556]
VAL_PRECISION Mean: 0.543 | Std: 0.011 | 95% CI: [0.523, 0.561]
Epoch 14/50



100%|██████████| 301/301 [01:38<00:00,  3.06it/s]


VAL_LOSS     0.048
VAL_ACC      Mean: 57.330 | Std: 1.185 | 95% CI: [55.346, 59.224]
VAL_KAPPA    Mean: 0.816 | Std: 0.011 | 95% CI: [0.797, 0.834]
VAL_F1       Mean: 0.520 | Std: 0.013 | 95% CI: [0.498, 0.540]
VAL_RECALL   Mean: 0.523 | Std: 0.012 | 95% CI: [0.501, 0.542]
VAL_PRECISION Mean: 0.566 | Std: 0.013 | 95% CI: [0.544, 0.586]

Early stopping at epoch 14. No improvement for 5 epochs.
Best epoch: 9 with kappa: 0.8231


## Test with gamma=2.0

In [8]:
model.load_state_dict(torch.load("models/focal-loss-gamma2.pth"))
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print("\n=== TEST RESULTS (Focal Loss gamma=2.0) ===")
print(result)

100%|██████████| 266/266 [01:18<00:00,  3.39it/s]



=== TEST RESULTS (Focal Loss gamma=2.0) ===
VAL_ACC      Mean: 59.138 | Std: 1.202 | 95% CI: [57.095, 60.933]
VAL_KAPPA    Mean: 0.831 | Std: 0.011 | 95% CI: [0.812, 0.849]
VAL_F1       Mean: 0.530 | Std: 0.013 | 95% CI: [0.509, 0.550]
VAL_RECALL   Mean: 0.536 | Std: 0.013 | 95% CI: [0.515, 0.556]
VAL_PRECISION Mean: 0.589 | Std: 0.013 | 95% CI: [0.567, 0.611]


## Training with Focal Loss (gamma=3.0 - focus more on hard examples)

In [ ]:
# Reload model
load_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

# Use Focal Loss with gamma=3.0 (focus more on hard examples)
loss_function = FocalLoss(alpha=0.25, gamma=3.0)

optimizer = optim.Adam(model.parameters(), lr=init_lr/warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier=warmup_factor, total_epoch=warmup_epochs, after_scheduler=scheduler_cosine)

print("\n=== Training with Focal Loss (gamma=3.0) ===")
train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/focal-loss-gamma3.txt",
    path_to_save_model="models/focal-loss-gamma3.pth",
    patience=5,
)


=== Training with Focal Loss (gamma=3.0) ===
Epoch 1/50



100%|██████████| 301/301 [01:39<00:00,  3.01it/s]


VAL_LOSS     0.010
VAL_ACC      Mean: 24.652 | Std: 1.026 | 95% CI: [22.989, 26.371]
VAL_KAPPA    Mean: 0.547 | Std: 0.011 | 95% CI: [0.528, 0.564]
VAL_F1       Mean: 0.186 | Std: 0.008 | 95% CI: [0.173, 0.199]
VAL_RECALL   Mean: 0.287 | Std: 0.009 | 95% CI: [0.272, 0.300]
VAL_PRECISION Mean: 0.174 | Std: 0.010 | 95% CI: [0.158, 0.191]
Salvando o melhor modelo... 0.0 -> 0.5466482447625342
Epoch 2/50



100%|██████████| 301/301 [01:38<00:00,  3.07it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.010
VAL_ACC      Mean: 20.059 | Std: 0.939 | 95% CI: [18.560, 21.607]
VAL_KAPPA    Mean: 0.513 | Std: 0.011 | 95% CI: [0.494, 0.531]
VAL_F1       Mean: 0.146 | Std: 0.007 | 95% CI: [0.135, 0.158]
VAL_RECALL   Mean: 0.250 | Std: 0.008 | 95% CI: [0.237, 0.263]
VAL_PRECISION Mean: 0.133 | Std: 0.010 | 95% CI: [0.117, 0.150]
Epoch 3/50



...

100%|██████████| 301/301 [01:39<00:00,  3.03it/s]


VAL_LOSS     0.024
VAL_ACC      Mean: 41.736 | Std: 1.212 | 95% CI: [39.831, 43.712]
VAL_KAPPA    Mean: 0.641 | Std: 0.012 | 95% CI: [0.622, 0.660]
VAL_F1       Mean: 0.291 | Std: 0.008 | 95% CI: [0.278, 0.304]
VAL_RECALL   Mean: 0.329 | Std: 0.009 | 95% CI: [0.314, 0.344]
VAL_PRECISION Mean: 0.288 | Std: 0.009 | 95% CI: [0.274, 0.304]
Salvando o melhor modelo... 0.6042009097457672 -> 0.6412845140055068
Epoch 5/50



100%|██████████| 301/301 [01:41<00:00,  2.97it/s]


VAL_LOSS     0.020
VAL_ACC      Mean: 46.840 | Std: 1.157 | 95% CI: [45.042, 48.809]
VAL_KAPPA    Mean: 0.711 | Std: 0.012 | 95% CI: [0.691, 0.731]
VAL_F1       Mean: 0.372 | Std: 0.010 | 95% CI: [0.356, 0.388]
VAL_RECALL   Mean: 0.409 | Std: 0.010 | 95% CI: [0.392, 0.426]
VAL_PRECISION Mean: 0.540 | Std: 0.038 | 95% CI: [0.410, 0.565]
Salvando o melhor modelo... 0.6412845140055068 -> 0.7111519249358708
Epoch 6/50



loss: 0.00151, smooth loss: 0.00259:  52%|█████▏    | 629/1204 [06:17<06:04,  1.58it/s]

## Test with gamma=3.0

In [ ]:
model.load_state_dict(torch.load("models/focal-loss-gamma3.pth"))
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print("\n=== TEST RESULTS (Focal Loss gamma=3.0) ===")
print(result)

## Training with Adaptive Focal Loss

In [ ]:
# Reload model
load_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

# Use Adaptive Focal Loss
loss_function = AdaptiveFocalLoss(gamma=2.0)

optimizer = optim.Adam(model.parameters(), lr=init_lr/warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier=warmup_factor, total_epoch=warmup_epochs, after_scheduler=scheduler_cosine)

print("\n=== Training with Adaptive Focal Loss ===")
train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/focal-loss-adaptive.txt",
    path_to_save_model="models/focal-loss-adaptive.pth",
    patience=5,
)

print(f"\nLearned alpha value: {torch.sigmoid(loss_function.alpha).item():.4f}")

## Test with Adaptive Focal Loss

In [ ]:
model.load_state_dict(torch.load("models/focal-loss-adaptive.pth"))
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print("\n=== TEST RESULTS (Adaptive Focal Loss) ===")
print(result)